# 03_trace_tasks (Kaggle CPU, competition attached, Internet OFF)
Traces all 129 tasks (pre = base+test_patch, post = base+gold+test_patch) and computes gold nodes.
Private notebook only (competition data rule 4b). Attach: the competition + private dataset `darkedges-code` (contains `src/`).
Resumable: re-run the same cells; finished (task, phase) pairs are skipped. To continue in a NEW session, attach the previous
version's output as an input and set `PRIOR_DIR` below.

In [ ]:
import sys, glob, os, json, time, platform
from pathlib import Path
# ---- locate the code dataset (contains src/tracelens and src/darkedges)
cands = glob.glob('/kaggle/input/**/src/tracelens', recursive=True)
assert cands, 'attach the darkedges-code dataset (src/ with tracelens + darkedges)'
SRC = str(Path(cands[0]).parent); sys.path.insert(0, SRC)
import importlib, tracelens
from darkedges import tracing, envbuild
print('python', platform.python_version(), '| src =', SRC)
loc = tracing.locate()                       # finds tasks.jsonl, snapshots/, graphs/, wheels/, docker/
for k, v in loc.items(): print(f'{k:10s} {v}')
OUT  = Path('/kaggle/working/out');  WORK = Path('/kaggle/temp/work');  WORK.mkdir(parents=True, exist_ok=True)
PRIOR_DIR = None                             # e.g. '/kaggle/input/03-trace-tasks/out' when resuming in a new session

In [ ]:
# ---- SMOKE TEST (T1 acceptance): pre must show >=1 failed test, post 0 failed, gold nodes non-empty.
# Picks fastapi_14492 (known spike task) + the first task of every other repo. Edit SMOKE to use your spike ids if you prefer.
_t = sorted(tracing.load_tasks(loc['tasks']), key=lambda t: t['instance_id'])
SMOKE = [t['instance_id'] for t in _t if t['instance_id'] == 'fastapi_14492']
for _r in sorted({tracing.repo_short(t) for t in _t}):
    SMOKE.append(next(t['instance_id'] for t in _t if tracing.repo_short(t) == _r and t['instance_id'] not in SMOKE))
print('smoke ids:', SMOKE)
SMOKE_OUT = Path('/kaggle/working/smoke')
tracing.run_all(loc, SMOKE_OUT, WORK, ids=SMOKE, retry_failed=True)
print(open(SMOKE_OUT/'traces'/'_status.jsonl').read())
print(open(SMOKE_OUT/'results'/'gold_nodes.jsonl').read())
print(open(SMOKE_OUT/'traces'/'_env.jsonl').read())


**Check before the full run:** for each smoke task `pre` should show `ok=True` with `tests_failed>=1`, `post` `tests_failed=0`, and gold_node_ids non-empty.
If `error` says *no tests recorded ... plugin inactive* the pytest plugin is not auto-loading: set `tracing.PYTEST_ARGS = ['-q','-p','no:cacheprovider','-p','tracelens.pytest_plugin']`.
If you see anyio plugin errors, add `'-p','no:anyio'` (as in the handoff example command). Then re-run the cell (`retry_failed=True`).

In [ ]:
# ---- FULL RUN (sequential; ~2.8 h estimate). Use shard=(i, n) to split across notebook versions, e.g. shard=(0, 2).
tracing.run_all(loc, OUT, WORK, shard=(0, 1), time_budget_h=11.0, retry_failed=False, prior_dir=PRIOR_DIR)

In [ ]:
# ---- GATE REPORT + PACK (run after the full run, or after each shard)
rep = tracing.validity_report(OUT, loc['tasks'], gate=100)
import zipfile
with zipfile.ZipFile('/kaggle/working/phase1_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in (OUT/'traces').iterdir():  z.write(f, f'data/traces/{f.name}')        # unzip at the repo root
    for f in (OUT/'results').iterdir(): z.write(f, f'results/{f.name}')
print(os.path.getsize('/kaggle/working/phase1_outputs.zip')/1e6, 'MB')

In [ ]:
# ---- OPTIONAL: failure triage (what to fix if valid < 100)
rows = [json.loads(l) for l in open(OUT/'traces'/'_valid.jsonl')]
from collections import Counter
for r in [r for r in rows if not r['valid']][:40]: print(r['instance_id'], '|', (r['reason'] or '')[:140])
envs = [json.loads(l) for l in open(OUT/'traces'/'_env.jsonl')]
print(Counter(n.split(':')[0] for e in envs for n in e['notes']).most_common(10))